This notebooks includes all experiments included in Section DSNU vs PRNU of the article. 

This code provides the following results: 

1. PRNU vs DSNU using dim images
2. PRNU vs DSNU statistics

## PRNU vs DSNU using dim images

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import rawpy
from scipy.stats import pearsonr
from multiprocessing import Pool, cpu_count


from numpy.fft import fft2, ifft2
from scipy.ndimage import filters
from tqdm import tqdm
import imageio

import sys
sys.path.append("./prnu")
import prnu

In [ ]:
def get_prnu(images):
    
    accumulator = np.zeros_like(images[0])

    for i, I_obs in enumerate(images):
        
        kernel_size = 10
        
        image_sa = np.zeros_like(I_obs, dtype = 'float64')

        red = I_obs[0::2, 0::2].astype('float64')
        red_sky = cv2.blur(red.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[0::2, 0::2] = (red-red_sky)

        green1 = I_obs[0::2, 1::2].astype('float64')
        green1_sky = cv2.blur(green1.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[0::2, 1::2]= (green1-green1_sky)

        green2 = I_obs[1::2, 0::2].astype('float64')
        green2_sky = cv2.blur(green2.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[1::2, 0::2] = (green2-green2_sky)         

        blue = I_obs[1::2, 1::2].astype('float64')
        blue_sky = cv2.blur(blue.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[1::2, 1::2] = (blue-blue_sky) 
        
        accumulator = image_sa + accumulator
    
    output = (accumulator/5).astype('float64')
    
    return output

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
def open_image(img_path):
    with rawpy.imread(img_path) as img:
        return img.raw_image_visible.astype(np.float32)

In [ ]:
sensors = ["A9", "B3", "C0", "D0", "E0"]
crop_size = (2823, 4240)

In [ ]:
def load_dark_sets(sensor_path):
    dark_path = os.path.join(sensor_path, "dark/raw")
    files = sorted(os.listdir(dark_path))
    
    dark_A = files[:5]
    dark_B = files[5:10]
    
    def load_list(file_list):
        imgs = []
        for f in file_list:
            img = open_image(os.path.join(dark_path, f))
            img = cut_fixed(img,  crop_size)
            imgs.append(img)
        return np.array(imgs)
    
    return load_list(dark_A), load_list(dark_B)

In [ ]:
def load_images_from_folder(folder, n=5):
    files = sorted(os.listdir(folder))[:n]
    images = []
    for f in files:
        img = open_image(os.path.join(folder, f))
        img = cut_fixed(img, crop_size)
        images.append(img)
    return np.array(images)

In [ ]:
def compute_dsnu(sensor_path):
    dark_path = os.path.join(sensor_path, "dark/raw")
    dark_imgs = load_images_from_folder(dark_path, n=5)
    dsnu = np.mean(dark_imgs, axis=0)
    return dsnu

In [ ]:
def compute_prnu_per_dim(sensor_path):
    prnu_dict = {}
    
    for folder in os.listdir(sensor_path):
        if folder.startswith("dim_"):
            dim_level = int(folder.split("_")[1])
            dim_path = os.path.join(sensor_path, folder, 'raw')
            
            imgs = load_images_from_folder(dim_path, n=5)
            fingerprint = get_prnu(imgs)
            
            prnu_dict[dim_level] = fingerprint
            
    return prnu_dict

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return np.corrcoef(a_flat, b_flat)[0,1]

def compute_pce(fingerprint, image):
    cc2d = prnu.crosscorr_2d(fingerprint, image)
    return prnu.pce(cc2d)['pce']

In [ ]:
results = {}

for sensor in sensors:
    sensor_path = os.path.join(base_path, sensor)
    
    dark_A, dark_B = load_dark_sets(sensor_path)

    dsnu = np.mean(dark_A, axis=0)
    dark_fingerprint = get_prnu(dark_B)
    
    prnu_dims = compute_prnu_per_dim(sensor_path)
    
    results[sensor] = {
        "matching": {"pearson": [], "pce": [], "dim": []},
        "mismatching": {"pearson": [], "pce": [], "dim": []}
    }
    
    # MATCHING
    for dim, prnu_f in prnu_dims.items():
        p = pearson_corr(dsnu, prnu_f)
        pce = compute_pce(dsnu, prnu_f)
        
        results[sensor]["matching"]["pearson"].append(p)
        results[sensor]["matching"]["pce"].append(pce)
        results[sensor]["matching"]["dim"].append(dim)
        
    p = pearson_corr(dsnu, dark_fingerprint)
    pce = compute_pce(dsnu, dark_fingerprint)

    results[sensor]["matching"]["pearson"].append(p)
    results[sensor]["matching"]["pce"].append(pce)
    results[sensor]["matching"]["dim"].append(np.inf)
    
    # MISMATCHING
    for other_sensor in sensors:
        if other_sensor == sensor:
            continue
            
        other_path = os.path.join(base_path, other_sensor)
        other_prnu = compute_prnu_per_dim(other_path)
        
        for dim, prnu_f in other_prnu.items():
            p = pearson_corr(dsnu, prnu_f)
            pce = compute_pce(dsnu, prnu_f)
            
            results[sensor]["mismatching"]["pearson"].append(p)
            results[sensor]["mismatching"]["pce"].append(pce)
            results[sensor]["mismatching"]["dim"].append(dim)
        
        dark_A_other, dark_B_other = load_dark_sets(other_path)
        dark_other = get_prnu(dark_B_other)

        p = pearson_corr(dsnu, dark_other)
        pce = compute_pce(dsnu, dark_other)

        results[sensor]["mismatching"]["pearson"].append(p)
        results[sensor]["mismatching"]["pce"].append(pce)
        results[sensor]["mismatching"]["dim"].append(np.inf)

In [ ]:
np.savez("pearson-lux-dsnu.npz", results=results)

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import rawpy
from scipy.stats import pearsonr
from multiprocessing import Pool, cpu_count


from numpy.fft import fft2, ifft2
from scipy.ndimage import filters
from tqdm import tqdm
import imageio

import sys
sys.path.append("./prnu")
import prnu

In [ ]:
def get_prnu(images):
    
    accumulator = np.zeros_like(images[0])

    for i, I_obs in enumerate(images):
        
        kernel_size = 10
        
        image_sa = np.zeros_like(I_obs, dtype = 'float64')

        red = I_obs[0::2, 0::2].astype('float64')
        red_sky = cv2.blur(red.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[0::2, 0::2] = (red-red_sky)

        green1 = I_obs[0::2, 1::2].astype('float64')
        green1_sky = cv2.blur(green1.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[0::2, 1::2]= (green1-green1_sky)

        green2 = I_obs[1::2, 0::2].astype('float64')
        green2_sky = cv2.blur(green2.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[1::2, 0::2] = (green2-green2_sky)         

        blue = I_obs[1::2, 1::2].astype('float64')
        blue_sky = cv2.blur(blue.copy(), (kernel_size, kernel_size)).astype('float64')
        image_sa[1::2, 1::2] = (blue-blue_sky) 
        
        accumulator = image_sa + accumulator
    
    output = (accumulator/5).astype('float64')
    
    return output

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
def open_image(img_path):
    with rawpy.imread(img_path) as img:
        return img.raw_image_visible.astype(np.float32)

In [ ]:
sensors = ["A9", "B3", "C0", "D0", "E0"]
crop_size = (2823, 4240)

In [ ]:
def load_flat_images(sensor_path, n=5):
    flat_path = os.path.join(sensor_path, "flat/raw")
    files = sorted(os.listdir(flat_path))[:n]
    
    images = []
    for f in files:
        img = open_image(os.path.join(flat_path, f))
        img = cut_fixed(img, crop_size)
        images.append(img)
    
    return np.array(images)

In [ ]:
def load_dark_sets(sensor_path):
    dark_path = os.path.join(sensor_path, "dark/raw")
    files = sorted(os.listdir(dark_path))
    
    dark_A = files[:5]
    dark_B = files[5:10]
    
    def load_list(file_list):
        imgs = []
        for f in file_list:
            img = open_image(os.path.join(dark_path, f))
            img = cut_fixed(img,  crop_size)
            imgs.append(img)
        return np.array(imgs)
    
    return load_list(dark_A), load_list(dark_B)

In [ ]:
def load_images_from_folder(folder, n=5):
    files = sorted(os.listdir(folder))[:n]
    images = []
    for f in files:
        img = open_image(os.path.join(folder, f))
        img = cut_fixed(img, crop_size)
        images.append(img)
    return np.array(images)

In [ ]:
def compute_dsnu(sensor_path):
    dark_path = os.path.join(sensor_path, "dark/raw")
    dark_imgs = load_images_from_folder(dark_path, n=5)
    dsnu = np.mean(dark_imgs, axis=0)
    return dsnu

In [ ]:
def compute_prnu_per_dim(sensor_path):
    prnu_dict = {}
    
    for folder in os.listdir(sensor_path):
        if folder.startswith("dim_"):
            dim_level = int(folder.split("_")[1])
            dim_path = os.path.join(sensor_path, folder, 'raw')
            
            imgs = load_images_from_folder(dim_path, n=5)
            fingerprint = get_prnu(imgs)
            
            prnu_dict[dim_level] = fingerprint
            
    return prnu_dict

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return np.corrcoef(a_flat, b_flat)[0,1]

def compute_pce(fingerprint, image):
    cc2d = prnu.crosscorr_2d(fingerprint, image)
    return prnu.pce(cc2d)['pce']

In [ ]:
results = {}

for sensor in sensors:
    
    sensor_path = os.path.join(base_path, sensor)
    
    flat_imgs = load_flat_images(sensor_path, n=5)
    reference_prnu = get_prnu(flat_imgs)
    
    prnu_dims = compute_prnu_per_dim(sensor_path)
    
    dark_A, dark_B = load_dark_sets(sensor_path)
    dark_fingerprint = get_prnu(dark_B)
    

    results[sensor] = {
        "matching": {"pearson": [], "pce": [], "dim": []},
        "mismatching": {"pearson": [], "pce": [], "dim": []}
    }
    
    for dim, prnu_f in prnu_dims.items():
        
        p = pearson_corr(reference_prnu, prnu_f)
        pce = compute_pce(reference_prnu, prnu_f)
        
        results[sensor]["matching"]["pearson"].append(p)
        results[sensor]["matching"]["pce"].append(pce)
        results[sensor]["matching"]["dim"].append(dim)
    
    p = pearson_corr(reference_prnu, dark_fingerprint)
    pce = compute_pce(reference_prnu, dark_fingerprint)
    
    results[sensor]["matching"]["pearson"].append(p)
    results[sensor]["matching"]["pce"].append(pce)
    results[sensor]["matching"]["dim"].append(np.inf)
    
    
    for other_sensor in sensors:
        if other_sensor == sensor:
            continue
        
        other_path = os.path.join(base_path, other_sensor)
        
        other_prnu_dims = compute_prnu_per_dim(other_path)
        
        for dim, prnu_f in other_prnu_dims.items():
            
            p = pearson_corr(reference_prnu, prnu_f)
            pce = compute_pce(reference_prnu, prnu_f)
            
            results[sensor]["mismatching"]["pearson"].append(p)
            results[sensor]["mismatching"]["pce"].append(pce)
            results[sensor]["mismatching"]["dim"].append(dim)
        
        dark_A_other, dark_B_other = load_dark_sets(other_path)
        dark_other = get_prnu(dark_B_other)
        
        p = pearson_corr(reference_prnu, dark_other)
        pce = compute_pce(reference_prnu, dark_other)
        
        results[sensor]["mismatching"]["pearson"].append(p)
        results[sensor]["mismatching"]["pce"].append(pce)
        results[sensor]["mismatching"]["dim"].append(np.inf)

In [ ]:
np.savez("pearson-lux-prnu.npz", results=results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_results(sensor, results_prnu, results_dsnu, save_dir=None):
    
    prnu = results_prnu[sensor]
    dsnu = results_dsnu[sensor]

    def prepare(dim_list, values):
        x = []
        labels = []
        
        for d in dim_list:
            if np.isinf(d):
                x.append(0)
                labels.append("dark")
            else:
                x.append(1.0 / d)
                labels.append(str(d))
        
        sorted_data = sorted(zip(x, values, labels), key=lambda t: t[0])
        x, y, labels = zip(*sorted_data)
        
        return list(x), list(y), list(labels)

    # ===== PRNU =====
    x_pm, y_pm, l_pm = prepare(
        prnu["matching"]["dim"],
        prnu["matching"]["pearson"]
    )
    
    x_pmis, y_pmis, l_pmis = prepare(
        prnu["mismatching"]["dim"],
        prnu["mismatching"]["pearson"]
    )

    # ===== DSNU =====
    x_dm, y_dm, l_dm = prepare(
        dsnu["matching"]["dim"],
        dsnu["matching"]["pearson"]
    )
    
    x_dmis, y_dmis, l_dmis = prepare(
        dsnu["mismatching"]["dim"],
        dsnu["mismatching"]["pearson"]
    )

    # ===== TICKS COMBINADOS =====
    all_x = x_pm + x_pmis + x_dm + x_dmis
    all_labels = l_pm + l_pmis + l_dm + l_dmis

    xtick_dict = {}
    for x, l in zip(all_x, all_labels):
        if x not in xtick_dict:
            xtick_dict[x] = l

    xticks = sorted(xtick_dict.keys())
    xticklabels = [xtick_dict[x] for x in xticks]

    max_labels = 8
    step = max(1, len(xticks) // max_labels)

    xticks_reduced = xticks[::step]
    xticklabels_reduced = xticklabels[::step]

    # ===== PLOT =====
    plt.figure(figsize=(9,5))

    # PRNU
    plt.plot(x_pm, y_pm, 'o-', label="PRNU Matching")
    plt.plot(x_pmis, y_pmis, 'x--', label="PRNU Mismatching")

    # DSNU
    plt.plot(x_dm, y_dm, 'o-', label="DSNU Matching")
    plt.plot(x_dmis, y_dmis, 'x--', label="DSNU Mismatching")

    plt.xticks(xticks_reduced, xticklabels_reduced, rotation=45)

    plt.xlabel("Dim level (dark = ∞)")
    plt.ylabel("Pearson")
    plt.title(f"{sensor} - PRNU vs DSNU")

    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    # ===== SAVE =====
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        out_path = save_dir / f"{sensor}_PRNU_DSNU.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close()

In [ ]:
prnu_data = np.load("pearson-lux-plus-point-prnu.npz", allow_pickle=True)
dsnu_data = np.load("pearson-lux-plus-point.npz", allow_pickle=True)

results_prnu = prnu_data["results"].item()
results_dsnu = dsnu_data["results"].item()

sensors = list(results_prnu.keys())

for sensor in sensors:
    plot_results(
        sensor,
        results_prnu=results_prnu,
        results_dsnu=results_dsnu,
        save_dir="plots-pearson-lux"
    )

## PRNU vs DSNU statistics

In [ ]:
import os
import numpy as np
import cv2
import rawpy

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
def open_image(img_path):
    with rawpy.imread(img_path) as img:
        return img.raw_image_visible.astype(np.float32)

In [ ]:
folders = [
    "A0","A1","A2","A3","A4","A5","A6","A7","A8","A9",
    "B0","B1","B2","B3","B4","B5","B6","B7","B8","B9",
    "C0","C1","D0","D1","E0","E1"
]
crop_size = (2823, 4240)

In [ ]:
def load_images_from_folder(folder_path, max_images=100):
    files = sorted(os.listdir(folder_path))[:max_images]
    images = []
    
    shape_ref = None
    
    for f in files:
        path = os.path.join(folder_path, f)
        
        img = cut_fixed(open_image(path), crop_size)
        
        if shape_ref is None:
            shape_ref = img.shape
        else:
            assert img.shape == shape_ref, "Shape problem"
        
        images.append(img)
    
    return np.array(images)

In [ ]:
def get_prnu(images):
    accumulator = np.zeros_like(images[0])

    for I_obs in images:
        
        kernel_size = 10
        image_sa = np.zeros_like(I_obs, dtype='float64')

        red = I_obs[0::2, 0::2].astype('float64')
        red_sky = cv2.blur(red.copy(), (kernel_size, kernel_size))
        image_sa[0::2, 0::2] = (red - red_sky)

        green1 = I_obs[0::2, 1::2].astype('float64')
        green1_sky = cv2.blur(green1.copy(), (kernel_size, kernel_size))
        image_sa[0::2, 1::2] = (green1 - green1_sky)

        green2 = I_obs[1::2, 0::2].astype('float64')
        green2_sky = cv2.blur(green2.copy(), (kernel_size, kernel_size))
        image_sa[1::2, 0::2] = (green2 - green2_sky)

        blue = I_obs[1::2, 1::2].astype('float64')
        blue_sky = cv2.blur(blue.copy(), (kernel_size, kernel_size))
        image_sa[1::2, 1::2] = (blue - blue_sky)

        accumulator += image_sa
    
    return accumulator / len(images)

In [ ]:
def compute_dsnu(images):
    return np.mean(images, axis=0)

In [ ]:
def pearson_corr(a, b):
    a = a.flatten()
    b = b.flatten()
    return np.corrcoef(a, b)[0, 1]

In [ ]:
data = {}

for folder in folders:
    print(f"Processing {folder}...")

    dark_path = os.path.join(BASE_PATH, folder, "dark/raw")
    flat_path = os.path.join(BASE_PATH, folder, "flat/raw")

    darks = load_images_from_folder(dark_path, 100)
    flats = load_images_from_folder(flat_path, 100)

    # --- splits ---
    darks_1, darks_2 = darks[:50], darks[50:100]
    flats_1, flats_2 = flats[:50], flats[50:100]

    # --- DSNU ---
    DSNU_1 = compute_dsnu(darks_1)
    DSNU_2 = compute_dsnu(darks_2)

    # --- PRNU ---
    PRNU_1 = get_prnu(flats_1)
    PRNU_2 = get_prnu(flats_2)

    model = folder[0]

    data[folder] = {
        "model": model,
        "DSNU": [DSNU_1, DSNU_2],
        "PRNU": [PRNU_1, PRNU_2]
    }

In [ ]:
def valid_pairs(a_list, b_list, same_object):
    pairs = []
    for i, a in enumerate(a_list):
        for j, b in enumerate(b_list):
            if same_object and i == j:
                continue
            pairs.append((a, b))
    return pairs

In [ ]:
results = {
    "DSNU_DSNU": {"same_camera": [], "same_model": [], "different_model": []},
    "PRNU_PRNU": {"same_camera": [], "same_model": [], "different_model": []},
    "PRNU_DSNU": {"same_camera": [], "same_model": [], "different_model": []},
}

cams = list(data.keys())

for i in range(len(cams)):
    for j in range(i, len(cams)):

        c1, c2 = cams[i], cams[j]
        m1, m2 = data[c1]["model"], data[c2]["model"]

        # label
        if c1 == c2:
            label = "same_camera"
        elif m1 == m2:
            label = "same_model"
        else:
            label = "different_model"

        # =====================
        # DSNU vs DSNU
        # =====================
        pairs = valid_pairs(
            data[c1]["DSNU"],
            data[c2]["DSNU"],
            same_object=(c1 == c2)
        )

        for a, b in pairs:
            results["DSNU_DSNU"][label].append(pearson_corr(a, b))

        # =====================
        # PRNU vs PRNU
        # =====================
        pairs = valid_pairs(
            data[c1]["PRNU"],
            data[c2]["PRNU"],
            same_object=(c1 == c2)
        )

        for a, b in pairs:
            results["PRNU_PRNU"][label].append(pearson_corr(a, b))

        # =====================
        # PRNU vs DSNU
        # =====================
        for p in data[c1]["PRNU"]:
            for d in data[c2]["DSNU"]:
                results["PRNU_DSNU"][label].append(pearson_corr(p, d))

In [ ]:
summary = {}

for k in results:
    summary[k] = {}
    for label in results[k]:
        vals = np.array(results[k][label])
        summary[k][label] = {
            "mean": np.mean(vals),
            "std": np.std(vals)
        }

In [ ]:
for k in summary:
    print("\n===", k, "===")
    for l in summary[k]:
        m = summary[k][l]["mean"]
        s = summary[k][l]["std"]
        print(l, f"{m:.4f} ± {s:.4f}")